# Credit Card Fraud Detection

## Project Overview

This notebook develops an end-to-end machine learning solution for detecting fraudulent credit card transactions.

The main challenge is **high class imbalance**, where fraudulent transactions represent only a very small portion of the dataset. The project therefore focuses on fraud-sensitive evaluation metrics such as **Precision, Recall, F1-Score, ROC-AUC, and PR-AUC**.

The final model is a **tuned XGBoost classifier** that is saved for deployment.

## Problem Statement

Develop a machine learning model to **detect fraudulent credit card transactions** by classifying each transaction as:

- `0` — Legitimate
- `1` — Fraudulent

The model should handle the highly imbalanced nature of the dataset and provide reliable fraud detection performance.

## Business Context

Credit card fraud can cause significant financial losses for banks and customers. An automated fraud detection system can analyze transaction information and identify suspicious transactions early, helping financial institutions reduce fraud losses and protect customers.

## Machine Learning Workflow

1. Import libraries and load the dataset
2. Inspect data structure and quality
3. Remove duplicate transactions
4. Analyze class imbalance
5. Perform exploratory data analysis
6. Engineer useful features
7. Separate features and target
8. Split data into training and testing sets
9. Scale the features without data leakage
10. Establish a Logistic Regression baseline
11. Handle imbalance with SMOTE
12. Compare Decision Tree, Random Forest, and XGBoost
13. Apply threshold-based predictions
14. Tune XGBoost hyperparameters
15. Evaluate the final model
16. Analyze feature importance
17. Save the trained model, scaler, and decision threshold

## 1. Import Libraries and Load Dataset

First, the required libraries are imported and the credit card transaction dataset is loaded.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

%matplotlib inline
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("../data/raw/credit_card.csv")
df.head()

## 2. Initial Data Inspection

Before modeling, we inspect the dataset dimensions, columns, missing values, duplicate rows, data types, unique values, and descriptive statistics.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum():,}")

### Remove Duplicate Transactions

Duplicate rows can cause repeated observations to influence model training. They are removed before continuing with the analysis.

In [ ]:
df.drop_duplicates(inplace=True)
print(f"Dataset shape after removing duplicates: {df.shape}")

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df.describe()

## 3. Analyze Class Imbalance

Fraud detection is an imbalanced classification problem. We first inspect the number and percentage of legitimate and fraudulent transactions.

In [ ]:
class_counts = df["Class"].value_counts()
class_percentages = df["Class"].value_counts(normalize=True) * 100

print("Class counts:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages)

In [ ]:
sns.countplot(data=df, x="Class")
plt.title("Class Distribution")
plt.xlabel("Class (0 = Legitimate, 1 = Fraudulent)")
plt.ylabel("Transaction Count")
plt.show()

## 4. Exploratory Data Analysis

The following visualizations help understand the distributions of transaction time, transaction amount, PCA-transformed features, and differences between legitimate and fraudulent transactions.

In [ ]:
df[["Time", "Amount"]].hist(figsize=(10, 4), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
df.loc[:, "V1":"V14"].hist(figsize=(8, 4), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
df.loc[:, "V15":"V28"].hist(figsize=(8, 4), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=df, x="Class", y="Amount")
plt.title("Transaction Amount: Legitimate vs Fraudulent")
plt.xlabel("Class (0 = Legitimate, 1 = Fraudulent)")
plt.ylabel("Amount")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Class", y="Time")
plt.title("Transaction Time: Legitimate vs Fraudulent")
plt.xlabel("Class (0 = Legitimate, 1 = Fraudulent)")
plt.ylabel("Time (seconds)")
plt.show()

### Compare PCA-Transformed Features by Class

The `V1` to `V28` variables are PCA-transformed features. Boxplots provide a visual comparison of their distributions across the two classes.

In [ ]:
v_features = df.loc[:, "V1":"V28"].columns

fig, axes = plt.subplots(4, 7, figsize=(22, 14))

for ax, col in zip(axes.flat, v_features):
    sns.boxplot(data=df, x="Class", y=col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

### Correlation Analysis

Correlation with the target can provide an initial indication of which variables are associated with fraud. Correlation alone is not used as the model-selection criterion.

In [ ]:
corr = df.corr()
corr

In [ ]:
class_corr = df.corr()["Class"].drop("Class").sort_values()

plt.figure(figsize=(8, 8))
class_corr.plot(kind="barh")
plt.title("Feature Correlation with Fraud")
plt.xlabel("Correlation")
plt.ylabel("Features")
plt.show()

## 5. Feature Engineering

Two additional features are created:

- `Hour`: extracts the hour of the day from the transaction time.
- `Amount_log`: applies `log1p` to reduce the effect of highly skewed transaction amounts.

In [ ]:
df["Hour"] = (df["Time"] // 3600) % 24
df["Amount_log"] = np.log1p(df["Amount"])

df.head()

## 6. Define Features and Target

`Class` is the dependent variable. All remaining columns are used as independent features.

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 7. Train-Test Split

The dataset is divided into training and testing sets using a **stratified split**.

Stratification is important here because the fraud class is very small and both sets should retain a similar class distribution.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

## 8. Feature Scaling

`StandardScaler` is fitted **only on the training data** and then used to transform both training and testing data.

This prevents information from the test set from leaking into the preprocessing step.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print("Training class distribution:")
print(y_train.value_counts())

print("\nTraining class percentages:")
print(y_train.value_counts(normalize=True) * 100)

## 9. Logistic Regression Baseline

Logistic Regression provides a simple baseline model.

Because the dataset is highly imbalanced, `class_weight="balanced"` is used so the minority fraud class receives greater importance during training.

In [ ]:
model = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

### Baseline Evaluation

The baseline is evaluated using fraud-sensitive metrics. In an imbalanced fraud problem, accuracy alone is not sufficient.

In [ ]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC:", average_precision_score(y_test, y_pred_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 10. SMOTE for Class Imbalance

SMOTE creates synthetic minority-class samples in the **training set**.

It is applied only to `X_train` and `y_train`. The original test set remains untouched so that evaluation reflects performance on unseen, naturally imbalanced data.

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Class distribution after SMOTE:")
print(y_train_smote.value_counts())

print("\nClass percentages after SMOTE:")
print(y_train_smote.value_counts(normalize=True) * 100)

In [ ]:
model_smote = LogisticRegression(random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test)
y_proba_smote = model_smote.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall:", recall_score(y_test, y_pred_smote))
print("F1-Score:", f1_score(y_test, y_pred_smote))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_smote))
print("PR-AUC:", average_precision_score(y_test, y_proba_smote))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

### Threshold Experiment with SMOTE Model

The default classification threshold is `0.50`. Here, a higher threshold of `0.95` is tested to make the model more conservative when labeling a transaction as fraud.

In [ ]:
smote_threshold = 0.95
y_pred_threshold = (y_proba_smote >= smote_threshold).astype(int)

print(classification_report(y_test, y_pred_threshold))

## 11. Decision Tree

A Decision Tree is tested as another classification approach. `class_weight="balanced"` is used to account for the minority fraud class.

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=15,
    class_weight="balanced",
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba_dt))
print("PR-AUC:", average_precision_score(y_test, y_proba_dt))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

## 12. Random Forest

Random Forest is evaluated as an ensemble tree-based model. The balanced class weight helps address the class imbalance during training.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("PR-AUC:", average_precision_score(y_test, y_proba_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

### Threshold Experiment with Random Forest

A threshold of `0.30` is tested to observe how changing the classification threshold affects the precision-recall trade-off.

In [ ]:
rf_threshold = 0.30
y_pred_threshold_rf = (y_proba_rf >= rf_threshold).astype(int)

print(classification_report(y_test, y_pred_threshold_rf))

## 13. XGBoost

XGBoost is evaluated as the main gradient-boosting approach.

The positive-class weight is calculated from the training data to account for the imbalance between legitimate and fraudulent transactions.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))
print("PR-AUC:", average_precision_score(y_test, y_proba_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

## 14. Hyperparameter Tuning

The XGBoost model is tuned using `GridSearchCV`.

Because this is a highly imbalanced fraud problem, **PR-AUC (`average_precision`)** is used as the tuning metric rather than accuracy.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

cv_xgb = GridSearchCV(
    xgb_model,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=2
)

cv_xgb.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(cv_xgb.best_params_)

print("\nBest Cross-Validation PR-AUC:")
print(cv_xgb.best_score_)

## 15. Evaluate the Tuned XGBoost Model

The best estimator found by cross-validation is evaluated on the untouched test set.

In [ ]:
best_xgb_model = cv_xgb.best_estimator_

y_pred_tuned_xgb = best_xgb_model.predict(X_test)
y_proba_tuned_xgb = best_xgb_model.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, y_pred_tuned_xgb))
print("Recall:", recall_score(y_test, y_pred_tuned_xgb))
print("F1-Score:", f1_score(y_test, y_pred_tuned_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned_xgb))
print("PR-AUC:", average_precision_score(y_test, y_proba_tuned_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned_xgb))

### Tuned Model Confusion Matrix

The confusion matrix shows the number of legitimate and fraudulent transactions that were correctly and incorrectly classified by the tuned XGBoost model.

In [ ]:
cm = confusion_matrix(y_test, y_pred_tuned_xgb)
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="coolwarm",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"]
)
plt.title("Confusion Matrix - Tuned XGBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 16. Feature Importance

Feature importance shows which input features contributed most to the tuned XGBoost model's predictions.

The `V1` to `V28` variables are PCA-transformed features, so their names do not directly represent business concepts such as customer age or merchant type.

In [ ]:
feature_names = df.drop(columns=["Class"]).columns

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": best_xgb_model.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

feature_importance

In [ ]:
plt.figure(figsize=(10, 8))

plt.barh(
    feature_importance["feature"],
    feature_importance["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Tuned XGBoost Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## 17. Final Decision Threshold

The deployed application uses a probability threshold of `0.50`.

A transaction is classified as fraudulent when its predicted fraud probability is greater than or equal to this threshold.

In [ ]:
final_threshold = 0.50

y_proba_best_xgb = best_xgb_model.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba_best_xgb >= final_threshold).astype(int)

print(classification_report(y_test, y_pred_final))

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="coolwarm",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"]
)

plt.title("Final Confusion Matrix - XGBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 18. Final Model Evaluation

These are the final test-set metrics used to summarize the deployed model's performance.

In [ ]:
print("Final Model Performance")
print("-" * 30)
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_best_xgb):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, y_proba_best_xgb):.4f}")

## 19. Save Model Artifacts

The trained XGBoost model, fitted scaler, and final decision threshold are saved so the same preprocessing and prediction logic can be reused by the Streamlit application.

Saving these artifacts also avoids retraining the model every time the application starts.

In [ ]:
joblib.dump(best_xgb_model, "../models/xgb_fraud_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(final_threshold, "../models/threshold.pkl")

print("Model artifacts saved successfully.")

## Conclusion

The project progressed from data inspection and exploratory analysis to preprocessing, imbalance handling, model comparison, XGBoost hyperparameter tuning, final evaluation, and model persistence.

The tuned XGBoost model is the selected model for deployment, and the saved artifacts are used by the Streamlit application for fraud prediction.

## Deployment

The trained model can now be used in the Streamlit application located at:

```text
app/app.py
```

The application loads the saved model and scaler and reproduces the feature engineering required during inference.